In [1]:
# ===== Imports and data =====
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/raw/student-mat.csv", sep=";")

In [2]:
# ===== Risk label from G3 =====
def assign_risk(g3):
    if g3 <= 9:
        return "High"
    elif g3 <= 13:
        return "Medium"
    return "Low"

RISK_ORDER = ["High", "Medium", "Low"]
df["risk"] = df["G3"].apply(assign_risk)

In [3]:
# ===== Feature groups =====
numeric_features = [
    "G1", "G2", "studytime", "absences", "failures",
    "famrel", "freetime", "goout", "Dalc", "Walc", "health", "traveltime"
]

binary_features = [
    "schoolsup", "famsup", "paid", "activities", "higher", "internet", "romantic"
]

feature_cols = numeric_features + binary_features

X = df[feature_cols]
y = df["risk"]

In [4]:
# ===== Stratified hold out split =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(y_train.value_counts().reindex(RISK_ORDER).to_dict())
print(y_test.value_counts().reindex(RISK_ORDER).to_dict())

Train: (316, 19)  Test: (79, 19)
{'High': 104, 'Medium': 132, 'Low': 80}
{'High': 26, 'Medium': 33, 'Low': 20}


In [5]:
# ===== Preprocessor, fitted later inside the model pipelines =====
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), binary_features),
    ]
)

In [6]:
# ===== Check the absences artifact =====
df["risk"] = df["G3"].apply(assign_risk)

print("Mean absences by risk category")
print(df.groupby("risk")["absences"].mean().round(2).reindex(RISK_ORDER))

print("\nShare of students with zero absences")
print("All students:", (df["absences"] == 0).mean().round(3))
print("Zero G3 students:", (df.loc[df["G3"] == 0, "absences"] == 0).mean().round(3))
print("Non zero G3 students with zero absences:",
      ((df["G3"] > 0) & (df["absences"] == 0)).sum())

Mean absences by risk category
risk
High      6.76
Medium    6.05
Low       3.78
Name: absences, dtype: float64

Share of students with zero absences
All students: 0.291
Zero G3 students: 1.0
Non zero G3 students with zero absences: 77


In [7]:
# ===== Sensitivity datasets =====
main_df = df.copy()
no_trivial_df = df[~((df["G3"] == 0) & (df["G2"] == 0))]
no_zero_df = df[df["G3"] > 0]

for name, d in [("main", main_df), ("no trivial", no_trivial_df), ("no zero G3", no_zero_df)]:
    print(name, len(d), d["risk"].value_counts().reindex(RISK_ORDER).to_dict())

main 395 {'High': 130, 'Medium': 165, 'Low': 100}
no trivial 382 {'High': 117, 'Medium': 165, 'Low': 100}
no zero G3 357 {'High': 92, 'Medium': 165, 'Low': 100}
